In [ ]:
#Upload data from google drive
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')
drive_path = '/content/drive/My Drive/Programacion/Actuary/freMTPL2/'
df = pd.read_csv(drive_path + 'mtpl_cleaned_portfolio.csv')

Mounted at /content/drive


In [ ]:
# ==============================================================================
# ENTRENAMIENTO DEL MODELO GLM POISSON (FRECUENCIA DE SINIESTROS)
# ==============================================================================
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split

# 1. DIVISIÓN DE DATOS (TRAIN / TEST)
# Separamos el 80% para entrenar el modelo y el 20% para probarlo después.
# Esto es vital para la futura comparación justa contra XGBoost.
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
print(f"Datos de Entrenamiento: {df_train.shape[0]} pólizas.")
print(f"Datos de Prueba (Test): {df_test.shape[0]} pólizas.")

# 2. DEFINICIÓN DE LA FÓRMULA MATEMÁTICA
# El operador '~' separa la variable objetivo (ClaimNb) de los predictores.
# El operador 'C()' transforma instantáneamente el texto en variables Dummies,
# tirando la primera categoría como base de referencia.
formula = (
    "ClaimNb ~ C(DrivAge_Group) + C(VehAge_Group) + C(Area) + "
    "Is_Diesel + BonusMalus_Grouped + VehPower_Capped"
)
print("\n--> Entrenando el GLM Poisson...")
# 3. ENTRENAMIENTO DEL MODELO
# Family = Poisson (Ideal para conteo de choques)
# Offset = log(Exposure) (Estandariza el riesgo para que sea "anualizado")
glm_freq = smf.glm(
    formula=formula,
    data=df_train,
    family=sm.families.Poisson(),
    offset=np.log(df_train['Exposure'])
).fit()

# 4. IMPRESIÓN DEL REPORTE ACTUARIAL
print("\n" + "="*80)
print("REPORTE DE RESULTADOS DEL MODELO (SUMMARY)")
print("="*80)
print(glm_freq.summary())

Datos de Entrenamiento: 531689 pólizas.
Datos de Prueba (Test): 132923 pólizas.

--> Entrenando el GLM Poisson...

REPORTE DE RESULTADOS DEL MODELO (SUMMARY)
                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               531689
Model:                            GLM   Df Residuals:                   531672
Model Family:                 Poisson   Df Model:                           16
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -85878.
Date:                Sun, 24 May 2026   Deviance:                   1.3119e+05
Time:                        16:10:31   Pearson chi2:                 9.34e+05
No. Iterations:                     7   Pseudo R-squ. (CS):            0.01037
Covariance Type:            nonrobust                                         
                                coef    std err     

In [ ]:
# ==============================================================================
# ENTRENAMIENTO DEL MODELO GLM GAMMA (SEVERIDAD DE SINIESTROS)
# ==============================================================================
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

print("--> Preparando los datos para el modelo de Severidad...")

# 1. FILTRAR SOLO PÓLIZAS CON SINIESTROS
# El modelo Gamma no acepta ceros; modelamos el costo medio de los que chocaron.
df_train_sev = df_train[df_train['TotalClaimAmount'] > 0].copy()
df_test_sev = df_test[df_test['TotalClaimAmount'] > 0].copy()

# 2. CALCULAR LA SEVERIDAD MEDIA POR PÓLIZA
# Dividimos el monto acotado por la cantidad de choques de esa póliza
df_train_sev['AvgSeverity'] = df_train_sev['ClaimAmount_Capped'] / df_train_sev['ClaimNb']
df_test_sev['AvgSeverity'] = df_test_sev['ClaimAmount_Capped'] / df_test_sev['ClaimNb']

print(f"Pólizas disponibles para entrenar Severidad: {df_train_sev.shape[0]}")

# 3. DEFINICIÓN DE LA FÓRMULA
# Nota actuarial: Se suele excluir BonusMalus de la severidad, ya que el historial
# de manejo afecta la probabilidad de chocar (frecuencia), pero no necesariamente
# el costo del daño material del auto una vez que el choque ya ocurrió.
formula_sev = (
    "AvgSeverity ~ C(DrivAge_Group) + C(VehAge_Group) + C(Area) + "
    "Is_Diesel + VehPower_Capped"
)

print("\n--> Entrenando el GLM Gamma...")
# 4. ENTRENAMIENTO DEL MODELO
# Especificamos la familia Gamma y forzamos el enlace Log para mantener la multiplicación
glm_sev = smf.glm(
    formula=formula_sev,
    data=df_train_sev,
    family=sm.families.Gamma(link=sm.families.links.Log())
).fit()

# 5. REPORTE DE RESULTADOS
print("\n" + "="*80)
print("REPORTE DE RESULTADOS - GLM SEVERIDAD (GAMMA)")
print("="*80)
print(glm_sev.summary())

--> Preparando los datos para el modelo de Severidad...
Pólizas disponibles para entrenar Severidad: 19926

--> Entrenando el GLM Gamma...

REPORTE DE RESULTADOS - GLM SEVERIDAD (GAMMA)
                 Generalized Linear Model Regression Results                  
Dep. Variable:            AvgSeverity   No. Observations:                19926
Model:                            GLM   Df Residuals:                    19910
Model Family:                   Gamma   Df Model:                           15
Link Function:                    Log   Scale:                          3.5343
Method:                          IRLS   Log-Likelihood:            -1.7538e+05
Date:                Sun, 24 May 2026   Deviance:                       22577.
Time:                        16:10:32   Pearson chi2:                 7.04e+04
No. Iterations:                    12   Pseudo R-squ. (CS):           0.003111
Covariance Type:            nonrobust                                         
                        

In [ ]:
# ==============================================================================
# EVALUACIÓN FINAL: CÁLCULO DE LA PRIMA PURA (PURE PREMIUM)
# ==============================================================================
import numpy as np

print("--> Proyectando tarifas en el dataset de Test...")

# 1. PREDECIR FRECUENCIA (Poisson)
# IMPORTANTE: Le tenemos que volver a pasar el logaritmo de la exposición
# para que el modelo sepa por cuánto tiempo cotizar a estos clientes.
df_test['Pred_Freq'] = glm_freq.predict(
    df_test,
    offset=np.log(df_test['Exposure'])
)

# 2. PREDECIR SEVERIDAD (Gamma)
# Calculamos el costo medio esperado para TODOS los clientes del test.
df_test['Pred_Sev'] = glm_sev.predict(df_test)

# 3. CALCULAR LA PRIMA PURA TÉCNICA (Tarifa Base)
# Frecuencia Esperada * Severidad Esperada
df_test['Pred_Pure_Premium'] = df_test['Pred_Freq'] * df_test['Pred_Sev']

# 4. EVALUACIÓN DE ADECUACIÓN DE CARTERA (Total Expected vs Actual)
# Esta es la métrica de negocio más importante: ¿La tarifa recauda lo suficiente?
total_actual_cost = df_test['ClaimAmount_Capped'].sum()
total_predicted_premium = df_test['Pred_Pure_Premium'].sum()
ratio_cobertura = total_predicted_premium / total_actual_cost

print("\n" + "="*60)
print("🎯 RESULTADOS DEL PORTAFOLIO DE PRUEBA (TEST SET)")
print("="*60)
print(f"Total Siniestros Reales a Pagar: € {total_actual_cost:,.2f}")
print(f"Total Primas Recaudadas (Modelo): € {total_predicted_premium:,.2f}")
print(f"Ratio de Cobertura (Tarifa/Costo): {ratio_cobertura:.2%}")
print("="*60)

# Mostrar un par de clientes aleatorios para ver sus tarifas
print("\nEjemplo de Tarifas por Cliente:")
df_test[['DrivAge_Group', 'Area', 'Is_Diesel', 'BonusMalus', 'Pred_Freq', 'Pred_Sev', 'Pred_Pure_Premium']].head(5)

--> Proyectando tarifas en el dataset de Test...

🎯 RESULTADOS DEL PORTAFOLIO DE PRUEBA (TEST SET)
Total Siniestros Reales a Pagar: € 9,259,380.45
Total Primas Recaudadas (Modelo): € 8,802,631.83
Ratio de Cobertura (Tarifa/Costo): 95.07%

Ejemplo de Tarifas por Cliente:


,DrivAge_Group,Area,Is_Diesel,BonusMalus,Pred_Freq,Pred_Sev,Pred_Pure_Premium
644757,55-64,D,0,50,0.030643,1625.517232,49.810572
147199,55-64,A,1,50,0.021147,1507.700857,31.883145
395273,35-44,D,1,51,0.028937,1526.950804,44.184703
631222,25-34,D,1,90,0.006438,1501.421624,9.665957
428397,45-54,D,1,50,0.021208,1667.886754,35.372879
